# Foundation EDA: IEEE-CIS Fraud Detection

Before any of the modeling machinery can be designed, we need to understand the
terrain. This notebook is that first survey. It does three things: it summarizes
how the dataset's columns distribute across types, it identifies the handful of
features that genuinely carry an ordinal interpretation, and it sets down a
written narrative meant to orient the feature profiler and the feature segmenter.
Along the way it explains *why*
[`config/ordinal_features.yaml`](../config/ordinal_features.yaml) contains what
it contains, since that file is the durable artifact this exploration produces.

A word on where the data lives. The raw CSVs sit under `data/raw/` once
`data/ieee-fraud-detection.zip` has been extracted; both the archive and the
extracted files are gitignored, so the top-level [README.md](../README.md)
documents the Kaggle download for anyone reproducing this. The corpus is large:
roughly 590,540 transactions, with `train_identity` attaching to only about a
24% subset by `TransactionID`. That partial join shapes a good deal of what
follows.

It is worth being explicit about what this notebook is *not*. It is a one-time
orientation pass, not the feature profiler itself; the feature profiler rebuilds
this work as a reproducible script that caches its output to
`outputs/feature_profile.json`. The aim here is deliberately narrow. We want just
enough evidence to defend the ordinal annotation and to point the next two
modules in the right direction, not an exhaustive data dictionary.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('..') / 'data' / 'raw'

df_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
df_id = pd.read_csv(DATA_DIR / 'train_identity.csv')
df = df_tx.merge(df_id, on='TransactionID', how='left')

print(f'transaction shape: {df_tx.shape}')
print(f'identity shape:    {df_id.shape}')
print(f'merged shape:      {df.shape}')
print(f'isFraud prevalence: {df_tx["isFraud"].mean():.4%}')

## Type distribution

The merged frame carries 434 columns, and they fall into three uneven groups.
The overwhelming majority, roughly 399 columns, are `float64`. This is where
the anonymized numeric features `V1..V339` live, alongside the time-delta
features `D1..D15`, the counts `C1..C14`, and the address and distance columns.
A much smaller set of about 31 columns is stored as strings, and this is the
categorical landscape we actually have to encode. The remainder, about 4
columns, are `int64`: `TransactionID`, `isFraud`, `TransactionDT`, and `card1`.

The practical takeaway is that the encoding-selection problem is far smaller than
the column count suggests. It reduces, essentially, to those 31 string columns
plus a handful of low-cardinality numerics. Everything else is already numeric
and needs no encoding decision at all.

In [ ]:
print(df.dtypes.value_counts())
print()
print(f'overall missingness: {df.isna().mean().mean():.2%}')
print(f'columns with >50% missing: {(df.isna().mean() > 0.5).sum()}')

## Categorical landscape (31 string columns)

Within those 31 string columns, cardinality is sharply bimodal: they cluster at
the two extremes rather than spreading evenly. At the low end, about 22 columns
hold five or fewer distinct values. These are the boolean-ish flags such as
`M1`–`M9` and `id_35`–`id_38`, together with small enumerations like
`ProductCD`, `card4`, `card6`, and `DeviceType`. At the high end sits a long tail
of identifier-like columns: `id_30` carries 75 OS strings, `id_31` carries 130
browser strings, `id_33` carries 260 screen resolutions, `DeviceInfo` carries
1,786 device strings, and the email-domain columns hold roughly 60 values each.

This split is not just descriptive; it anticipates a decision. When high
cardinality coincides with heavy missingness (as it does for several of these
columns), a full one-hot expansion becomes both wasteful and brittle. That
pressure is exactly what will push the encoding evaluator toward grouped or
frequency encoding for the worst offenders rather than a naive per-value scheme.

In [ ]:
str_cols = df.select_dtypes(include='object').columns.tolist()
rows = []
for c in str_cols:
    rows.append({
        'column': c,
        'nunique': df[c].nunique(dropna=True),
        'missing_rate': df[c].isna().mean(),
    })
cat_summary = pd.DataFrame(rows).sort_values('nunique').reset_index(drop=True)
cat_summary

## Ordinal candidate investigation

The default assumption for these categoricals should be that they are nominal,
and for good reason: anonymization stripped semantic ordering from nearly
everything in the dataset. The interesting question is therefore where that
default breaks down. The only credible exceptions are columns whose values
*literally encode* an ordering in their own string form, where the order is a
property of the data rather than a guess we impose on it.

Two columns clear that bar:

- **`id_34`**. Its values are `match_status:-1`, `match_status:0`,
  `match_status:1`, and `match_status:2`. The integer suffix supplies the
  ordering directly.
- **`M4`**. Its values are `M0`, `M1`, and `M2`. Again the integer suffix points
  to a monotone level. The IEEE-CIS data dictionary never documents what that
  ordering *means*, but the surface form is enough to make the ordinal reading
  defensible on its own terms.

Two further columns are close calls that we deliberately *exclude*, letting them
fall back to the 0.30 ordinal rubric score downstream (the
"uncertain-ordering penalty"):

- **`id_23`** (IP proxy types: HIDDEN / ANONYMOUS / TRANSPARENT). One could
  argue for an ordering by degree of transparency, but the direction (does more
  concealment imply higher fraud risk?) is a modeling judgment, not something
  the values themselves assert. We would rather let the MILP pay the 0.30
  penalty than impose an order we cannot defend.
- **`id_15`** (Found / New / Unknown). A confidence-style sequence is plausible
  here too, yet the surface form simply does not commit to one direction.

The boolean `T`/`F` columns (`M1`–`M3`, `M5`–`M9`, and `id_35`–`id_38`) deserve
a brief note, because they look like they might belong in this discussion and do
not. They are *technically* orderable as `{F, T}`, but for a two-value
categorical a one-hot encoding and an ordinal encoding collapse to the same
single column. Annotating them as ordinal would leave the encoding evaluator's
decision matrix unchanged, so there is nothing to gain by doing it.

In [ ]:
for c in ['id_34', 'M4', 'id_23', 'id_15']:
    print(f'=== {c} ===')
    print(df[c].value_counts(dropna=False).to_string())
    print()

## Conclusions feeding the next deliverables

**Ordinal annotation (`config/ordinal_features.yaml`).** The investigation above
resolves to two entries, and only two:

```yaml
id_34:
  - "match_status:-1"
  - "match_status:0"
  - "match_status:1"
  - "match_status:2"
M4:
  - M0
  - M1
  - M2
```

Every other categorical falls back to the unlisted 0.30 score in the encoding
evaluator's rubric. That is not an oversight but the intended behavior: the
rubric penalty is where our honest uncertainty about ordering gets encoded, and
most of these columns earn it.

**Orienting the feature profiler.** The profiler will emit, for each column, a
detected type (numeric or categorical), its cardinality, a summary of its
distribution shape, its missingness rate, and its mutual information against
`isFraud` computed via `_mutual_info_classifier`. This exploration surfaces three
considerations the profiler should carry forward:

- Scale forces a compromise. The merged frame is 590,540 × 434, and running
  `_mutual_info_classifier` over all of it is expensive. The profiler should
  subsample using the `RANDOM_SEED`; a stratified 10% sample keeps it consistent
  with the encoding-evaluator policy and still yields stable MI rankings.
- The MI value is a *characterization* signal and nothing more. LOF (the anomaly
  detector) and the representation-analysis MLPs never see `isFraud` as a
  training target. This is the invariant set out in [CLAUDE.md](../CLAUDE.md),
  and the module docstring must restate it so the boundary is impossible to miss.
- Missingness is structural here, not incidental. Many columns exceed 50% missing,
  so the profiler should report the missingness rate as a first-class field. That
  lets the segmenter and the encoding evaluator route such columns deliberately;
  for instance, a missingness indicator paired with passthrough rather than a
  full encoding sweep.

**Orienting the feature segmenter.** The plan fixes five domain-labeled
segments (*transaction amount*, *identity/device*, *behavioral frequency*,
*temporal/timing*, and *card/account*), and the categorical landscape mapped out
above slots into them cleanly. `TransactionAmt` and the `V*` aggregates belong to
transaction-amount; `DeviceType`, `DeviceInfo`, and `id_30`/`id_31`/`id_33` to
identity/device; the count columns `C1..C14` to behavioral-frequency;
`TransactionDT` and `D1..D15` to temporal/timing; and `card1..card6`, `addr1`,
and `addr2` to card/account. A rule-based assignment will therefore cover the
bulk of the frame on its own, leaving k-means on the feature-profile vector to
handle only the residual that the rules do not reach.